<a href="https://colab.research.google.com/github/jacoaji02/speech-ai-model-learning/blob/main/model_training_using_huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
sentence = "my name is ajin"
tokens = tokenizer.tokenize(sentence)
encoded_input = tokenizer(sentence)
input_ids = encoded_input["input_ids"]
attention_mask = encoded_input["attention_mask"]

# Print the requested outputs
print("--- Single Sentence Analysis ---")
print(f"Original Text:   {sentence}")
print(f"Tokens:          {tokens}")
print(f"Input IDs:       {input_ids}")
print(f"Attention Mask:  {attention_mask}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

--- Single Sentence Analysis ---
Original Text:   my name is ajin
Tokens:          ['my', 'name', 'is', 'aj', '##in']
Input IDs:       [101, 2026, 2171, 2003, 19128, 2378, 102]
Attention Mask:  [1, 1, 1, 1, 1, 1, 1]


In [ ]:
batch_sentences = [
    "I love machine learning",
    "Deep learning is incredibly fast and fun",
    "Python programming"
]

encoded_input = tokenizer(batch_sentences, padding=True, truncation=True, max_length = 8, return_tensors="pt")
print("input_id_tensor", encoded_input["input_ids"])
print("attention_mask_tensor", encoded_input["attention_mask"])

input_id_tensor tensor([[  101,  1045,  2293,  3698,  4083,   102,     0,     0],
        [  101,  2784,  4083,  2003, 11757,  3435,  1998,   102],
        [  101, 18750,  4730,   102,     0,     0,     0,     0]])
attention_mask_tensor tensor([[1, 1, 1, 1, 1, 1, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 0, 0, 0, 0]])


In [ ]:
from transformers import AutoTokenizer
import torch

raw_sentences = [
    "This machine learning course is absolutely amazing",
    "I love programming deep learning models with python",
    "This bad coding tutorial is incredibly frustrating",
    "I hate fixing broken bugs and errors"
]

# TODO:
# 1. Load the 'bert-base-uncased' tokenizer using AutoTokenizer.
# 2. Tokenize 'raw_sentences' with padding=True, truncation=True, max_length=8, and return_tensors="pt".
# 3. Extract the "input_ids" from the result and assign it to a variable named X_train.

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
embeded_input = tokenizer(raw_sentences, padding=True, truncation=True, max_length=8, return_tensors="pt")
X_train = embeded_input['input_ids']


# TEST: Run this cell. If it prints a [4, 8] matrix shape, you pass!
print("X_train Shape:", X_train.shape if X_train is not None else "Not implemented yet")

X_train Shape: torch.Size([4, 8])


In [ ]:
import torch.nn as nn

class RealTextClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        # TODO:
        # 1. Create a nn.Embedding layer named 'self.embeddings' using vocab_size and emb_dim.
        # 2. Create a nn.Linear layer named 'self.classifier' that maps from emb_dim to 1 output value.
        # 3. Set up a nn.Sigmoid() layer named 'self.sigmoid'.
        self.embeddings = nn.Embedding(vocab_size, emb_dim)
        self.classifier = nn.Linear(emb_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # Leave this blank for now, we will build this in Exercise 3!
        pass

# TEST: Testing your structural initialization
try:
    test_model = RealTextClassifier(vocab_size=30522, emb_dim=16)
    print("Success: Layers initialized perfectly!")
except Exception as e:
    print("Error:", e)

Success: Layers initialized perfectly!


In [ ]:
import torch

class RealTextClassifier(nn.Module):
    def __init__(self, vocab_size, emb_dim):
        super().__init__()
        # Paste your working initialization layers here:
        self.embeddings = nn.Embedding(vocab_size, emb_dim)
        self.classifier = nn.Linear(emb_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # TODO:
        # 1. Pass the inputs 'x' through your embedding layer.
        #    Input Shape: [Batch_Size, Sequence_Length] -> Output Shape: [Batch_Size, Sequence_Length, Emb_Dim]
        x = self.embeddings(x)

        # 2. Average the vectors across the sequence length dimension (dim=1) to get one vector per sentence.
        #    Hint: Use torch.mean(x, dim=1)
        #    Output Shape: [Batch_Size, Emb_Dim]
        x = torch.mean(x, dim=1)

        # 3. Pass that condensed vector through your classifier linear layer,
        #    then pass that output through your sigmoid layer, and return it!
        x = self.classifier(x)

        output = self.sigmoid(x)
        return output

# TEST: Run this to see if a batch of token IDs can pass completely through your network!
try:
    model = RealTextClassifier(vocab_size=30522, emb_dim=16)
    mock_batch = torch.randint(0, 30522, (4, 8)) # Mock input matrix of shape [4, 8]
    predictions = model(mock_batch)
    print("Success! Output Shape is:", predictions.shape) # Should be torch.Size([4, 1])
except Exception as e:
    print("Error during execution:", e)

Success! Output Shape is: torch.Size([4, 1])


In [ ]:
import torch.nn as nn
import torch.optim as optim

# Instantiate the model architecture
vocab_pool_size = 30522
model = RealTextClassifier(vocab_size=vocab_pool_size, emb_dim=16)

# TODO:
# 1. Define Binary Cross Entropy Loss (nn.BCELoss) and name it 'criterion'.
# 2. Define an Adam Optimizer (optim.Adam) named 'optimizer'.
#    Pass it the model's parameters (model.parameters()) and set the learning rate (lr) to 0.05.

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

# TEST: Run this cell. If it passes without error, your configuration is set!
try:
    print("Criterion:", type(criterion).__name__)
    print("Optimizer:", type(optimizer).__name__)
except Exception as e:
    print("Error:", e)

Criterion: BCELoss
Optimizer: Adam


In [ ]:
import torch
torch.manual_seed(42)

# Raw Target labels from Exercise 1 (2 positive sentences, 2 negative sentences)
labels = torch.tensor([[1.0], [1.0], [0.0], [0.0]], dtype=torch.float32)


print("--- Starting 30 Epochs of Optimization ---")
for epoch in range(30):
    # TODO: Implement the classic 5-step PyTorch training routine inside this loop:
    # 1. Clear out old historical gradients using your optimizer object
    # 2. Pass X_train through the model to get its predictions
    # 3. Compute the loss using your criterion against the target labels
    # 4. Perform backpropagation (loss.backward) to calculate weight updates
    # 5. Tell the optimizer to take a step forward and apply those weight changes

    # [Write your 5 lines of loop code here]
    optimizer.zero_grad()
    predictions = model(X_train)
    loss = criterion(predictions, labels)
    loss.backward()
    optimizer.step()

    # Track the progress every 5 epochs
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}/30 | Loss: {loss.item():.4f}")

--- Starting 30 Epochs of Optimization ---
Epoch 5/30 | Loss: 0.4470
Epoch 10/30 | Loss: 0.1259
Epoch 15/30 | Loss: 0.0161
Epoch 20/30 | Loss: 0.0023
Epoch 25/30 | Loss: 0.0005
Epoch 30/30 | Loss: 0.0002


In [ ]:
# TODO:
# 1. Force the model into evaluation mode using model.eval().
#    This turns off temporary training behaviors like dropout layers.
#
# 2. Open a 'with torch.no_grad():' context block.
#    This completely freezes the gradient math tracking engine to save RAM memory during inference.
#
# 3. Inside the context block, pass X_train through the model to get your final probability decimals.
#    Assign it to the variable named 'final_probabilities'.

# [Write your validation setup code here]

model.eval()

with torch.no_grad():
  final_probabilities = model(X_train)


# --- Accuracy Calculation Engine (Pre-written for you!) ---
# 4. Convert probabilities to absolute binary decisions (0.0 or 1.0) with a 0.5 cutoff threshold
binary_predictions = (final_probabilities > 0.5).float()

# 5. Compute matches and divide by total samples to get accuracy
correct_matches = (binary_predictions == labels).sum().item()
accuracy = (correct_matches / len(labels)) * 100

print(f"🎯 Final Pipeline Evaluation Accuracy: {accuracy:.1f}%")

🎯 Final Pipeline Evaluation Accuracy: 100.0%
